Overall:

- first get community based timelined data (community - month)

# Fetch all needed data

In [ ]:
import json
import sys
processed_dir = "../processed/"
sys.path.append(processed_dir)

from data_utils import read_json

submission_id_to_entry_all = read_json(f"{processed_dir}/content_included/submission_id_to_entry_all.json")
submissions_with_english_reactions = set(read_json(f"{processed_dir}/content_included/submission_ids_with_english_reactions.json"))
article_title_to_content_all = read_json(f"{processed_dir}/content_included/article_title_to_content_all.json")
articles_in_english = set(read_json(f"{processed_dir}/content_included/articles_titles_in_english.json"))
english_article_titles_to_keyphrases = read_json(f"{processed_dir}/content_included/english_article_titles_to_keyphrases.json")

print(len(submission_id_to_entry_all))           # 168446 posts
print(len(submissions_with_english_reactions))   # 157228 en_posts
print(len(article_title_to_content_all))         # 142240 articles
print(len(articles_in_english))                  # 120584 en_articles
print(len(english_article_titles_to_keyphrases)) # 120584 en_articles

subreddit_to_submission_timeline = {} # {subreddit: {month: [(submission_id, date)]}}
for submission_id, entry in submission_id_to_entry_all.items():
    if submission_id in submissions_with_english_reactions:
        subreddit = entry["subreddit"]
        which_date = entry["post_creation_date"]
        which_month = which_date[:7]
        if subreddit not in subreddit_to_submission_timeline:
            subreddit_to_submission_timeline[subreddit] = {}
        if which_month not in subreddit_to_submission_timeline[subreddit]:
            subreddit_to_submission_timeline[subreddit][which_month] = []
        subreddit_to_submission_timeline[subreddit][which_month].append((submission_id, which_date))

for subreddit, month_to_submissions in subreddit_to_submission_timeline.items():
    for month, submissions in month_to_submissions.items():
        month_to_submissions[month] = sorted(submissions, key=lambda x: x[1])

# write to file
with open(f"./subreddit_to_submission_timeline.json", "w") as f:
    json.dump(subreddit_to_submission_timeline, f, indent=4)

keyError_count = 0

# all in one file
all_in_one_english = {}
for subreddit, month_to_submissions in subreddit_to_submission_timeline.items():
    if subreddit not in all_in_one_english:
        all_in_one_english[subreddit] = {}
    for month, submissions in month_to_submissions.items():
        if month not in all_in_one_english[subreddit]:
            all_in_one_english[subreddit][month] = []
        for (submission_id, which_date) in submissions:
            if submission_id in submissions_with_english_reactions:
                article_title = submission_id_to_entry_all[submission_id]["article_title"]
                article_content = article_title_to_content_all[article_title]
                if article_title in articles_in_english:
                    all_in_one_english[subreddit][month].append({
                        "submission_id": submission_id,
                        "submission_date": which_date,
                        "news_title": article_title,
                        "news_content": article_content,
                        "news_keywords": english_article_titles_to_keyphrases[article_title],
                        "reactions": submission_id_to_entry_all[submission_id]["top_reactions"]
                    })
                else:
                    # reactions are in English, but the article content wasn't detected as English
                    # it seems like the content wasn't of high quality. (short or missing thus wasn't detected)
                    keyError_count += 1
                    print(subreddit, "|", which_date, "|", submission_id)
                    print(f"TITLE:\n    {article_title}")
                    print(f"CONTENT:\n    {article_content[:100]}")
                    print("-----------------")

print(keyError_count) # 15733

print(len(all_in_one_english)) # subreddits: 4206
print(all_in_one_english.keys())
print(type(all_in_one_english["politics"]))
print(all_in_one_english["politics"].keys())
print(len(all_in_one_english["politics"]["2020-01"])) # 33
with open(f"./all_in_one_english.json", "w") as f:
    json.dump(all_in_one_english, f, indent=4)

168446
157220
142240
120584
120584
worldnews | 2020-01-19 | eqy8nx
TITLE:
    China says New coronavirus is ‘preventable and controllable’
CONTENT:
    
-----------------
worldnews | 2020-01-22 | esokmf
TITLE:
    Coronavirus has hit the U.S. but a local expert says it's not time to panic
CONTENT:
    Coronavirus has hit the U.S. but a local expert says it's not time to panic
(WRDW)
-----------------
worldnews | 2020-01-23 | et2bth
TITLE:
    China coronavirus: Wuhan medical staff being infected at much faster pace than reported as national death toll hits 26
CONTENT:
    
-----------------
worldnews | 2020-01-23 | et2ath
TITLE:
    Chinese expert who came down with Wuhan coronavirus after saying it was controllable thinks he was infected through his eyes
CONTENT:
    Wang Guangfa became infected after a visit to Wuhan. Photo: QQ.com
-----------------
worldnews | 2020-01-24 | ethioi
TITLE:
    Novel coronavirus 2019-nCoV: early estimation of epidemiological parameters and epidemic pred

# Select subreddits

In [ ]:
import os
import json
import sys

processed_dir = "../processed/"
sys.path.append(processed_dir)

from data_utils import read_json

all_in_one_english = read_json("all_in_one_english.json")
articles_in_english = set(read_json(f"{processed_dir}/content_included/articles_titles_in_english.json"))

# figure out a way to selected a bunch of STATISTICALLY interesting subreddits to base our analysis part of work on. 
# review: the structure of all_in_one_english: {subreddit: {month: [entries]}}

def month_to_index(yyyy_dash_mm_str):
    assert('-' in yyyy_dash_mm_str)
    return int(yyyy_dash_mm_str.split("-")[1])-1

monthly_counts = {}
for subreddit in all_in_one_english:
    monthly_counts[subreddit] = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    for month_str in all_in_one_english[subreddit]:
        month_idx = month_to_index(month_str)
        monthly_counts[subreddit][month_idx] = len(all_in_one_english[subreddit][month_str])

for subreddit in monthly_counts:
    print(subreddit, monthly_counts[subreddit])

# at least posted 2 times a month for 9 months
subreddit_monthly_requirements = {} # {subreddit: [1, 0, 1, 0, ...]}
monthly_minimum = 2
min_month = 9
for subreddit in monthly_counts:
    subreddit_monthly_requirements[subreddit] = [1 if count >= monthly_minimum else 0 for count in monthly_counts[subreddit]]

subreddit_satisfied_month = {subreddit for subreddit in subreddit_monthly_requirements if sum(subreddit_monthly_requirements[subreddit]) >= min_month}

print(len(subreddit_satisfied_month))

articles_posted_by = {}
for subreddit in all_in_one_english:
    for month in all_in_one_english[subreddit]:
        for entry in all_in_one_english[subreddit][month]:
            article_title = entry["news_title"]
            if article_title not in articles_posted_by:
                articles_posted_by[article_title] = []
            articles_posted_by[article_title].append(subreddit)

print(len(articles_posted_by))
multiple_post_count = 0
for article in articles_posted_by:
    if len(articles_posted_by[article]) > 1:
        multiple_post_count += 1
print(multiple_post_count)

subreddit_co_post_count = {}
for article in articles_posted_by:
    if len(articles_posted_by[article]) > 1:
        for subreddit in articles_posted_by[article]:
            subreddit_co_post_count[subreddit] = subreddit_co_post_count.get(subreddit, 0) + 1

# rank subreddits by the number of articles they have in common with other subreddits
sorted_subreddit_co_post_count = sorted(subreddit_co_post_count.items(), key=lambda x: x[1], reverse=True)
top_coposting_subreddits = set([subreddit for (subreddit, count) in list(sorted_subreddit_co_post_count)[:300]])

# interesectino of the two sets
print(len(subreddit_satisfied_month.intersection(top_coposting_subreddits)))
combined_set = subreddit_satisfied_month.intersection(top_coposting_subreddits)
sorted_combined_set = sorted(list(combined_set), key=str.casefold)

# wrtie to file line by line
with open(f"./subreddits-check-again.txt", "w") as f:
    for subreddit in sorted_combined_set:
        f.write(f"https://reddit.com/r/{subreddit}\n")


worldnews [250, 651, 1699, 1358, 701, 343, 363, 339, 232, 256, 271, 345]
collapse [7, 7, 83, 93, 37, 32, 22, 25, 13, 22, 30, 35]
news [67, 1, 15, 21, 16, 10, 8, 114, 96, 150, 153, 160]
Cebu [1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0]
Sino [12, 30, 87, 65, 44, 18, 18, 10, 16, 16, 8, 20]
australia [30, 54, 429, 273, 153, 75, 118, 130, 77, 69, 53, 59]
brisbane [4, 2, 17, 12, 1, 1, 7, 7, 4, 1, 2, 4]
canada [35, 79, 655, 547, 311, 171, 90, 133, 176, 141, 192, 165]
vancouver [7, 6, 138, 103, 59, 42, 31, 79, 55, 31, 80, 62]
China [14, 94, 113, 100, 28, 11, 8, 8, 7, 8, 13, 24]
metacanada [15, 12, 61, 63, 37, 9, 8, 0, 0, 0, 0, 0]
ID_News [3, 4, 11, 7, 1, 7, 3, 2, 4, 1, 1, 1]
EverythingScience [1, 7, 43, 56, 20, 11, 9, 17, 14, 16, 7, 17]
Thailand [1, 11, 14, 8, 2, 5, 3, 4, 5, 4, 0, 9]
todayilearned [1, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 0]
HongKong [13, 16, 15, 3, 3, 2, 7, 1, 3, 0, 0, 1]
politics [33, 459, 4186, 2949, 1654, 925, 1146, 911, 682, 1617, 736, 1069]
bestof [0, 2, 11, 2, 0, 1, 3, 1, 1, 1, 1, 3]
m

# Geo-based subreddits

In [3]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

processed_dir = "../processed/"
sys.path.append(processed_dir)

from data_utils import pretty_print, write_json




# Read the CSV files
geo_data = pd.read_csv('./geo-oriented.csv', names=["Reddit_url", "Continent", "Country_Abbr", "Region_Abbr", "Geo_Parent", "Geo_Type", "Topic"]).fillna('/')
geo_parent_is_global = geo_data[geo_data['Geo_Parent'] == "GLOBAL"] 
geo_parent_is_continental = geo_data[geo_data['Geo_Parent'] == geo_data["Continent"]]
geo_parent_is_country = geo_data[geo_data['Geo_Parent'] == geo_data["Country_Abbr"]]
geo_parent_not_above = geo_data[(geo_data['Geo_Parent'] != "GLOBAL") & (geo_data['Geo_Parent'] != geo_data["Continent"]) & (geo_data['Geo_Parent'] != geo_data["Country_Abbr"])]
# print(geo_parent_is_global)
# print("------------------------------------------------")
# print(geo_parent_is_continental)
# print("------------------------------------------------")
# print(geo_parent_is_country)
# print("------------------------------------------------")
# print(geo_parent_not_above)

# fig, axes = plt.subplots(nrows=1, ncols=3)
# pd.value_counts(geo_data['Continent']).plot(ax=axes[0]).bar()
# pd.value_counts(geo_data['Country_Abbr']).plot(ax=axes[1]).bar()
# pd.value_counts(geo_data['Geo_Type']).plot(ax=axes[2]).bar()
# geo_data['Continent'].value_counts().plot.bar()
# geo_data['Country_Abbr'].value_counts().plot.bar()
# geo_data['Geo_Type'].value_counts().plot.bar()
# plt.show()


geo_records = geo_data.to_dict('records')
geo_subreddits = {record['Reddit_url'].split("/")[-1].strip(): record for record in geo_records}
comm_to_topic = {subreddit: record["Topic"] for subreddit, record in geo_subreddits.items()}

geo_path_to_subreddits = {} # geo_path -> matching subreddits (if any, in a list)
subreddit_to_parent_geo_path = {} # subreddit -> parent's geo_path


for comm, record in geo_subreddits.items():
    geo_path_list = ["GLOBAL"] # root node
    geo_parent = record['Geo_Parent']
    # geo_parent must exist
    assert(geo_parent != "/")
    # our annotation requirements (regional_abbr must be "/" if the geo-comm is not the entire area of that region (in that case put abbr in geo_parent))
    assert(record['Region_Abbr'] != geo_parent) 
    # print("processing:", comm, record)
    
    if geo_parent == "GLOBAL": # Geo_Type == COUNTRY+ or (transcontinental) COUNTRY 
        # about "COUNTRY+" and "COUNTRY": 
        #       only GLOBAL or CONTINENTAL orgs beyond a sovereign country count as "COUNTRY+"; 
        #       COUNTRY of "united" states, kingdoms, etc don't count as "COUNTRY+"" but as "COUNTRY".
        assert(record["Geo_Type"] in ["COUNTRY+", "COUNTRY"])
        if record['Continent'] != "/": 
            geo_path_list.append(record['Continent'])
        else:
            # one might ask about https://en.wikipedia.org/wiki/List_of_transcontinental_countries 
            # but so far not found in our selected subreddits
            raise ValueError("Annotation Error or Global community within more-than-one/no continent (shall use '+' to annotate instead of '/'!).")
        geo_path = "-".join(geo_path_list)
        if geo_path in geo_path_to_subreddits:
            geo_path_to_subreddits[geo_path].append(comm)
        else:
            geo_path_to_subreddits[geo_path] = [comm]
        subreddit_to_parent_geo_path[comm] = "-".join(geo_path_list[:-1])
        # print("Global community:", comm, "\n\tGeo Path:", geo_path, "\n\tParent's Geo Path:", subreddit_to_parent_geo_path[comm])
        # print("--------------------------------")
        continue

    else: # Geo_Type: <= COUNTRY+
        assert(record['Continent'] != "/") 
        geo_path_list.append(record['Continent']) 
        # so far, the geo path is GLOBAL-CONTINENT
        
        if geo_parent == record['Continent']: # Geo_Type == COUNTRY or COUNTRY+
            assert(record['Country_Abbr'] != "/")
            assert(record["Geo_Type"] in ["COUNTRY", "COUNTRY+"])
            # could be "+" in the country abbreviation (e.g., "USA+CAN") => do nothing here (post-process later)
            geo_path_list.append(record['Country_Abbr'])
            # so far, the geo path is GLOBAL-CONTINENT-COUNTRY(+)
            geo_path = "-".join(geo_path_list)
            if geo_path in geo_path_to_subreddits:
                geo_path_to_subreddits[geo_path].append(comm)
            else:
                geo_path_to_subreddits[geo_path] = [comm]
            subreddit_to_parent_geo_path[comm] = "-".join(geo_path_list[:-1])
            # print("Country community:", comm, "\n\tGeo Path:", geo_path, "\n\tParent's Geo Path:", subreddit_to_parent_geo_path[comm])
            # print("--------------------------------")
            continue

        else: # Geo_Type: <= COUNTRY (why could it still be COUNTRY? read the disclaimer below)
            
            """
            Disclaimer: we labeled some regions included in our selected subreddits with a geolocation dewelling British Isles as COUNTRY according to wikipedia and some other online sources in English, but also followed the official govermental documents to accompany them with country and/or regional abbreviations (Region_Abbr). For instance, for its Region_Abbr, Country_Abbr, and Geo_type, subreddit /r/northernireland gets GB-NIR, GBR, COUNTRY. We hope this can be considered an acceptable practice and we mean no offense to any relevant parties. We are open to all suggestions to improve our annotation (being aware that wikipedia and other pages can be biases or outdated). 
            """
            
            # so far, the geo path is GLOBAL-CONTINENT
            assert(record['Country_Abbr'] != "/")
            geo_path_list.append(record['Country_Abbr'])
            # so far, the geo path is GLOBAL-CONTINENT-COUNTRY

            if geo_parent == record['Country_Abbr']:
                if record["Region_Abbr"] == "/": # its geo-parent is a country, but itself has no regional abbreviation (i.e. not within any state/province/district/etc.), thus must be some place special
                    # nothing like this yet in our selected subreddits
                    # TODO
                    subreddit_to_parent_geo_path[comm] = "-".join(geo_path_list)
                    raise ValueError("Special Case")

                else:
                    geo_path_list.append(record["Region_Abbr"])
                    # so far, the geo path is GLOBAL-CONTINENT-COUNTRY-STATE/PROVINCE_like
                    geo_path = "-".join(geo_path_list)
                    if geo_path in geo_path_to_subreddits:
                        geo_path_to_subreddits[geo_path].append(comm)
                    else:
                        geo_path_to_subreddits[geo_path] = [comm]
                    subreddit_to_parent_geo_path[comm] = "-".join(geo_path_list[:-1])
                    # print("State/Province/Regional community:", comm, "\n\tGeo Path:", geo_path, "\n\tParent's Geo Path:", subreddit_to_parent_geo_path[comm])
                    # print("--------------------------------")
            else: # Geo_Type: < STATE/PROVINCE_like (i.e. geo_parent has not appread in the geo path yet)
                # so far, the geo path is GLOBAL-CONTINENT-COUNTRY
                assert(record["Region_Abbr"] == "/") # if it's state/province/etc.-like, it must have a country as its parent, thus not falling into this if-else block
                geo_path_list.append(geo_parent)
                # so far, the geo path is GLOBAL-CONTINENT-COUNTRY-STATE/PROVINCE_like
                geo_path_list.append(comm)
                geo_path = "-".join(geo_path_list)
                if geo_path in geo_path_to_subreddits:
                    geo_path_to_subreddits[geo_path].append(comm)
                else:
                    geo_path_to_subreddits[geo_path] = [comm]
                subreddit_to_parent_geo_path[comm] = "-".join(geo_path_list[:-1])
                # print("City/City+ community:", comm, "\n\tGeo Path:", geo_path, "\n\tParent's Geo Path:", subreddit_to_parent_geo_path[comm])
                # print("--------------------------------")




# pretty_print(subreddit_to_parent_geo_path)
# pretty_print(geo_path_to_subreddits)

# handle the "+" in the country abbreviation

subreddit_to_geo_paths = {} # subreddit -> geo_paths_set

for geo_path, subreddits in geo_path_to_subreddits.items():
    for subreddit in subreddits:
        if subreddit not in subreddit_to_geo_paths:
            subreddit_to_geo_paths[subreddit] = []
        if "+" in geo_path:
            assert(geo_path.count("+") == 1)
            geo_path_splits = geo_path.split("-")
            for idx, split in enumerate(geo_path_splits):
                if "+" in split:
                    sub_splits = split.split("+")
                    # add both paths to the set
                    subreddit_to_geo_paths[subreddit].append("-".join(geo_path_splits[:idx] + [sub_splits[0]] + geo_path_splits[idx+1:]))
                    subreddit_to_geo_paths[subreddit].append("-".join(geo_path_splits[:idx] + [sub_splits[1]] + geo_path_splits[idx+1:]))

        else:
            subreddit_to_geo_paths[subreddit].append(geo_path)


subreddit_is_covid_specific = {comm: True if "COVID" in topic else False for comm, topic in comm_to_topic.items()}

pretty_print(subreddit_to_geo_paths)
pretty_print(subreddit_is_covid_specific)

# write to file
subreddit_geo_info = {
    "subreddit_to_geo_paths": subreddit_to_geo_paths,
    "subreddit_is_covid_specific": subreddit_is_covid_specific
}
write_json("subreddit_geo_info.json", subreddit_geo_info)

{
    "Alabama": [
        "GLOBAL-NORTH AMERICA-USA-AL"
    ],
    "alberta": [
        "GLOBAL-NORTH AMERICA-CAN-AB"
    ],
    "Arkansas": [
        "GLOBAL-NORTH AMERICA-USA-AR"
    ],
    "Austin": [
        "GLOBAL-NORTH AMERICA-USA-TX-Austin"
    ],
    "australia": [
        "GLOBAL-OCEANIA-AUS"
    ],
    "bayarea": [
        "GLOBAL-NORTH AMERICA-USA-CA-bayarea"
    ],
    "boston": [
        "GLOBAL-NORTH AMERICA-USA-MA-boston"
    ],
    "boulder": [
        "GLOBAL-NORTH AMERICA-USA-CO-boulder"
    ],
    "Brampton": [
        "GLOBAL-NORTH AMERICA-CAN-ON-Brampton"
    ],
    "brisbane": [
        "GLOBAL-OCEANIA-AUS-QLD-brisbane"
    ],
    "britishcolumbia": [
        "GLOBAL-NORTH AMERICA-CAN-BC"
    ],
    "Calgary": [
        "GLOBAL-NORTH AMERICA-CAN-AB-Calgary"
    ],
    "California": [
        "GLOBAL-NORTH AMERICA-USA-CA"
    ],
    "CoronavirusCA": [
        "GLOBAL-NORTH AMERICA-USA-CA"
    ],
    "canada": [
        "GLOBAL-NORTH AMERICA-CAN"
    ],
    "Canad

In [8]:
# # more analysis on the geo community structure
# # continents?
# contients = set()
# continents_to_geo_subreddits = {}
# for geo_path in geo_path_to_subreddits:
#     contient = geo_path.split("-")[:2]
#     contient = contient[0]+"-"+contient[1]
#     contients.add(contient)
#     subreddits = geo_path_to_subreddits[geo_path]
#     for subreddit in subreddits:
#         geo_type = geo_subreddits[subreddit]["Geo_Type"]
#         if contient not in continents_to_geo_subreddits:
#             continents_to_geo_subreddits[contient] = {}
#         continents_to_geo_subreddits[contient][subreddit] = geo_type

# for contient in continents_to_geo_subreddits:
#     print(contient, len(continents_to_geo_subreddits[contient]))
#     # print(continents_to_geo_subreddits[contient])
#     # reverse subreddit name and type
#     reversed_dict = {}
#     for subreddit, type in continents_to_geo_subreddits[contient].items():
#         if type not in reversed_dict:
#             reversed_dict[type] = set()
#         reversed_dict[type].add(subreddit)
#     for geo_type, subreddits in reversed_dict.items():
#         print(geo_type, len(subreddits), subreddits)
#     print("--------------------------------")
    
# # creating a balancing set but don't mix them, use as a separate test set
# # it seems like only the north america part of the world has state-like-level included in the top 207 subreddits.

# print(len(geo_subreddits))

# Topic-based subreddits

In [4]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

processed_dir = "../processed/"
sys.path.append(processed_dir)

from data_utils import pretty_print, write_json

topic_data = pd.read_csv('./topic-oriented.csv', names=["Reddit_url", "Geo_Estimate", "Topic"]).fillna('/')
topic_records = topic_data.to_dict('records')
topic_subreddits = {record['Reddit_url'].split("/")[-1].strip(): record for record in topic_records}
for subreddit, info in topic_subreddits.items():
    topic_subreddits[subreddit]["Topic"] = [info["Topic"].split("+")]
    topic_subreddits[subreddit].pop("Reddit_url")
pretty_print(topic_subreddits)

write_json("subreddits_topic_info.json", topic_subreddits)
# geo_est_to_subreddit = {}
# for subreddit, info in topic_subreddits.items():
#     geo_est = info["Geo_Estimate"]
#     if geo_est not in geo_est_to_subreddit:
#         geo_est_to_subreddit[geo_est] = set()
#     geo_est_to_subreddit[geo_est].add(subreddit)

# for geo_est, subreddits in geo_est_to_subreddit.items():
#     print(geo_est, len(subreddits))
#     print(subreddits)
#     print("---------------------------")
# # come up with an automatic way to merge this with geo-based communties?

# topic_to_subreddit = {}
# for subreddit, info in topic_subreddits.items():
#     topics = info["Topic"].split("+")
#     for topic in topics:
#         if topic not in topic_to_subreddit:
#             topic_to_subreddit[topic] = set()
#         topic_to_subreddit[topic].add(subreddit)
# print(topic_to_subreddit.keys())
# for topic, subreddits in topic_to_subreddit.items():
#     print(topic, len(subreddits))
#     print(subreddits)
#     print("---------------------------")

{
    "AnythingGoesNews": {
        "Geo_Estimate": "GLOBAL",
        "Topic": [
            [
                "SOCIAL_DISCUSSION_AND_THOUGHTS"
            ]
        ]
    },
    "apple": {
        "Geo_Estimate": "GLOBAL",
        "Topic": [
            [
                "TECHNOLOGY"
            ]
        ]
    },
    "asianamerican": {
        "Geo_Estimate": "GLOBAL",
        "Topic": [
            [
                "SOCIAL_DISCUSSION_AND_THOUGHTS"
            ]
        ]
    },
    "atheism": {
        "Geo_Estimate": "GLOBAL",
        "Topic": [
            [
                "RELIGION"
            ]
        ]
    },
    "AusFinance": {
        "Geo_Estimate": "AUS",
        "Topic": [
            [
                "BUSINESS/ECONOMICS/FINANCE"
            ]
        ]
    },
    "AustralianPolitics": {
        "Geo_Estimate": "AUS",
        "Topic": [
            [
                "POLITICS"
            ]
        ]
    },
    "aznidentity": {
        "Geo_Estimate": "GLOBAL",
      

# All-in-one for selected subreddits

In [ ]:
import json
import sys
processed_dir = "../processed/"
sys.path.append(processed_dir)

from data_utils import read_json, write_json

subreddit_to_monthly_record = read_json(f"all_in_one_english.json")
subreddit_to_geo_info = read_json(f"subreddit_geo_info.json")
subreddit_to_topic_info = read_json(f"subreddits_topic_info.json")

selected_subreddit_data = {}

for subreddit in subreddit_to_monthly_record:
    if subreddit in subreddit_to_geo_info["subreddit_to_geo_paths"]:
        selected_subreddit_data[subreddit] = {
            "subreddit_info": {
                "type": "geo",
                "info": {
                    "geo_path": subreddit_to_geo_info["subreddit_to_geo_paths"][subreddit],
                    "is_covid_specific": subreddit_to_geo_info["subreddit_is_covid_specific"][subreddit]
                },
            },
            "monthly_record": subreddit_to_monthly_record[subreddit]
        }
    if subreddit in subreddit_to_topic_info:
        selected_subreddit_data[subreddit] = {
            "subreddit_info": {
                "type": "topic",
                "info": subreddit_to_topic_info[subreddit]
            },
            "monthly_record": subreddit_to_monthly_record[subreddit]
        }

write_json("./comm_data.json", selected_subreddit_data)